# **7 - Hyperparameter Tuning (Optuna on XGBoost) — Colab**
Inputs: Day 4 split + Day 6 winning model (XGBoost, ROC-AUC 0.8851 val / 0.8854 test) + the canonical 78-feature set from `day7_model_features.csv`.
Goal: tune XGBoost hyperparameters with Optuna to beat the untuned Day 6 baseline, then do one honest final evaluation.

### **Overview & decisions**
- **Feature set is locked:** the 78 one-hot features in `day7_model_features.csv` (canonical schema for all downstream steps). `tract_minority_population_percent` is kept but flagged `is_race_proxy=1`.
- **Tuning target:** XGBoost (won Day 6). Objective = **PR-AUC** on `X_val` (imbalance-aware); ROC-AUC reported as secondary.
- **Trial speed:** each Optuna trial trains on a `TRIAL_SUB_N` stratified subsample (default 1M rows) for fast iteration; the final selected model is retrained on the full `X_train`.
- **Imbalance:** `scale_pos_weight = neg/pos` fixed (data property, not tuned).
- **Protocol:** tuning uses only `X_val` (no `X_test` leakage). The final selected model gets one `X_test` evaluation.
- **Robustness:** each trial is appended to `day7_optuna_progress.csv` so partial runs are still useful.
- **Env overrides:** `DAY7_TRIALS` (default 20) and `DAY7_TRIAL_SUB` (default 1000000).
### **Files to place in Google Drive (before running)**
This Colab notebook mounts Drive and reads everything from one folder. Copy these 8 files from your local `data/processed/modelling/` into a Drive folder (default assumed: `MyDrive/HMDA_Data/modelling/`) and adjust `M` in the setup cell if your folder name differs:
- `X_train.parquet`
- `X_val.parquet`
- `X_test.parquet`
- `y_train.parquet`
- `y_val.parquet`
- `y_test.parquet`
- `day7_model_features.csv`
- `day6_model_metrics.csv`

`X_train.parquet` is the largest (~hundreds of MB on disk, ~2.4 GB in memory); free Colab's ~12 GB RAM is sufficient. All generated artifacts are written back into that same Drive folder.


In [1]:
!pip install -q optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 10.2 MB/s eta 0:00:00


In [2]:
import os, time, json
from google.colab import drive
import numpy as np, pandas as pd
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score,
                             recall_score, f1_score, confusion_matrix, roc_curve, precision_recall_curve)
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
import xgboost as xgb

def _gpu_available():
    import warnings
    # xgboost >= 2.0: GPU via device='cuda' (+ tree_method='hist')
    try:
        with warnings.catch_warnings(record=True) as _w:
            warnings.simplefilter('always')
            _d = xgb.DMatrix(np.zeros((2, 2)), label=np.zeros(2))
            xgb.train({'tree_method': 'hist', 'device': 'cuda', 'objective': 'binary:logistic'}, _d, 1)
            _msgs = ' '.join(str(w.message) for w in _w)
            if 'couldn' + chr(39) + 't find any available GPU' in _msgs or 'GPU to CPU' in _msgs:
                raise RuntimeError('cuda requested but no GPU present')
        return True
    except Exception as e:
        print('GPU (device=cuda) check failed:', repr(e))
    # legacy xgboost < 2.0: tree_method='gpu_hist'
    try:
        _d = xgb.DMatrix(np.zeros((2, 2)), label=np.zeros(2))
        xgb.train({'tree_method': 'gpu_hist', 'objective': 'binary:logistic'}, _d, 1)
        return True
    except Exception as e:
        print('GPU (gpu_hist) check failed:', repr(e))
    return False

USE_GPU = _gpu_available()
_ver = tuple(int(p) for p in xgb.__version__.split('.')[:2])
if USE_GPU:
    if _ver >= (2, 0):
        TREE_METHOD, DEVICE = 'hist', 'cuda'
    else:
        TREE_METHOD, DEVICE = 'gpu_hist', None
else:
    TREE_METHOD, DEVICE = 'auto', None
N_JOBS = 1 if USE_GPU else -1
gpu_kwargs = {'device': DEVICE} if DEVICE else {}
print('GPU available:', USE_GPU, '| tree_method =', TREE_METHOD, '| device =', DEVICE)

# Mount Google Drive and point M at the folder holding the 8 data files
drive.mount('/content/drive')
# >>> Copy the 8 files (see 2nd markdown cell) into a Drive folder, e.g. MyDrive/HMDA_Data/modelling/
# >>> If your folder name differs, change the line below.
M = '/content/drive/MyDrive/HMDA_Data/modelling/'
FIG = M + 'figures/day7/'
os.makedirs(FIG, exist_ok=True)

N_TRIALS = int(os.environ.get('DAY7_TRIALS', '10'))
TRIAL_SUB_N = int(os.environ.get('DAY7_TRIAL_SUB', '1000000'))
print('N_TRIALS =', N_TRIALS, '| TRIAL_SUB_N =', TRIAL_SUB_N, '| optuna', optuna.__version__)

X_train = pd.read_parquet(M + 'X_train.parquet').astype(np.float32)
X_val   = pd.read_parquet(M + 'X_val.parquet').astype(np.float32)
X_test  = pd.read_parquet(M + 'X_test.parquet').astype(np.float32)
y_train = pd.read_parquet(M + 'y_train.parquet')['approved'].astype('int8').values
y_val   = pd.read_parquet(M + 'y_val.parquet')['approved'].astype('int8').values
y_test  = pd.read_parquet(M + 'y_test.parquet')['approved'].astype('int8').values
spw = float((y_train == 0).sum()) / float((y_train == 1).sum())
print('scale_pos_weight (neg/pos) =', round(spw, 4))


GPU available: True | tree_method = hist | device = cuda
Mounted at /content/drive
N_TRIALS = 10 | TRIAL_SUB_N = 1000000 | optuna 4.9.0
scale_pos_weight (neg/pos) = 0.3396


## **Step 1 - Load the canonical feature set**
Read `day7_model_features.csv` (the 78 features persisted in Step A). Assert it matches the data, then build the trial subsample used inside Optuna. Also define the shared `evaluate_model`.

In [3]:
feat = pd.read_csv(M + 'day7_model_features.csv')
assert len(feat) == 78, f"expected 78 features, got {len(feat)}"
FEATURES = feat['feature'].tolist()
assert set(FEATURES).issubset(set(X_train.columns)), "feature CSV has columns not in X_train"
print('loaded', len(FEATURES), 'model features | race-proxy flagged:',
      int(feat['is_race_proxy'].sum()))

# Trial subsample for fast Optuna iteration (fixed random_state for reproducibility)
Xsub, _, ysub, _ = train_test_split(X_train[FEATURES], y_train,
                                    train_size=TRIAL_SUB_N, random_state=42, stratify=y_train)
print('trial subsample:', Xsub.shape)

def evaluate_model(model, X, y, name='model'):
    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= 0.5).astype(int)
    r = dict(name=name, roc_auc=roc_auc_score(y, proba), pr_auc=average_precision_score(y, proba),
             precision=precision_score(y, pred), recall=recall_score(y, pred),
             f1=f1_score(y, pred), cm=confusion_matrix(y, pred), proba=proba)
    print(f"[{name}] ROC-AUC={r['roc_auc']:.4f} PR-AUC={r['pr_auc']:.4f} P={r['precision']:.4f} R={r['recall']:.4f} F1={r['f1']:.4f}")
    return r


loaded 78 model features | race-proxy flagged: 1
trial subsample: (1000000, 78)


## **Step 2 - Optuna objective**
Search over `max_depth`, `learning_rate`, `n_estimators`, `subsample`, `colsample_bytree`, `min_child_weight`, `reg_lambda`, `reg_alpha`. `scale_pos_weight` is fixed. Each trial fits XGBoost on the trial subsample (positional numpy, as in Day 6) with early stopping on `X_val`, and returns PR-AUC. Every trial is written to `day7_optuna_progress.csv`.

In [4]:
prog_csv = M + 'day7_optuna_progress.csv'
if os.path.exists(prog_csv):
    os.remove(prog_csv)

def log_trial(study, trial):
    row = {'trial': trial.number, 'value': trial.value, 'roc_auc': trial.user_attrs.get('roc_auc')}
    row.update(trial.params)
    pd.DataFrame([row]).to_csv(prog_csv, mode='a', header=not os.path.exists(prog_csv), index=False)

def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 1000, 3000, step=500),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-2, 10.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'scale_pos_weight': spw, 'importance_type': 'gain', 'eval_metric': 'auc', 'tree_method': TREE_METHOD, **gpu_kwargs,
        'random_state': 42, 'n_jobs': N_JOBS, 'early_stopping_rounds': 50,
    }
    Xtr = Xsub.values; Xva = X_val[FEATURES].values
    clf = XGBClassifier(**params)
    clf.fit(Xtr, ysub, eval_set=[(Xtr, ysub), (Xva, y_val)], verbose=False)
    proba = clf.predict_proba(Xva)[:, 1]
    pr = average_precision_score(y_val, proba)
    trial.set_user_attr('roc_auc', float(roc_auc_score(y_val, proba)))
    return pr

t0 = time.time()
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_TRIALS, callbacks=[log_trial])
print(f'Optuna done in {time.time()-t0:.1f}s | best PR-AUC={study.best_value:.4f}')
print('best params:'); print(json.dumps(study.best_params, indent=2))


[I 2026-08-28 17:15:38,114] A new study created in memory with name: no-name-d6ecfa70-b79b-4b22-a1b5-9f3ce299036f
/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [17:16:02] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
[I 2026-08-28 17:16:04,941] Trial 0 finished with value: 0.9210966205038716 and parameters: {'max_depth': 4, 'learning_rate': 0.002080274115160644, 'n_estimators': 1000, 'subsample': 0.9049622961885426, 'colsample_bytree': 0.5596577955911823, 'min_child_weight': 10, 'reg_lambda': 1.0237012494112574, 'reg_alpha': 0.009236580904634

Optuna done in 530.9s | best PR-AUC=0.9494
best params:
{
  "max_depth": 10,
  "learning_rate": 0.029393392247576217,
  "n_estimators": 3000,
  "subsample": 0.86637030299845,
  "colsample_bytree": 0.7602638701835958,
  "min_child_weight": 6,
  "reg_lambda": 0.012341225700071089,
  "reg_alpha": 0.06889775400590788
}


## Step 3 - Best parameters & final model
Persist the best hyperparameters to `day7_best_xgb_params.json`. Retrain the final XGBoost on the **full** `X_train` with those params (early stopping on `X_val`), then evaluate on `X_val` and the single reserved `X_test` (tuning never touched `X_test`).

In [6]:
import joblib

In [5]:
best_params = dict(study.best_params)
json.dump(best_params, open(M + 'day7_best_xgb_params.json', 'w'), indent=2)
print('saved day7_best_xgb_params.json')

final_params = dict(best_params)
final_params.update({'scale_pos_weight': spw, 'importance_type': 'gain', 'eval_metric': 'auc', 'tree_method': TREE_METHOD, **gpu_kwargs,
                     'random_state': 42, 'n_jobs': -1, 'early_stopping_rounds': 50})
Xtr_f = X_train[FEATURES].values; Xva_f = X_val[FEATURES].values
final = XGBClassifier(**final_params)
final.fit(Xtr_f, y_train, eval_set=[(Xtr_f, y_train), (Xva_f, y_val)], verbose=False)

print('final best_iteration:', final.best_iteration)
res_val = evaluate_model(final, Xva_f, y_val, 'XGB_tuned (X_val)')
res_test = evaluate_model(final, X_test[FEATURES].values, y_test, 'XGB_tuned (X_test)')


saved day7_best_xgb_params.json
final best_iteration: 2999
[XGB_tuned (X_val)] ROC-AUC=0.8958 PR-AUC=0.9543 P=0.9147 R=0.8620 F1=0.8876
[XGB_tuned (X_test)] ROC-AUC=0.8961 PR-AUC=0.9544 P=0.9148 R=0.8623 F1=0.8878


In [7]:
# persist the finetuned model (step 4)
_MODEL_PKL = M + 'day7_tuned_xgboost.pkl'
joblib.dump(final, _MODEL_PKL)
print('wrote', _MODEL_PKL)
# native XGBoost format (version-robust: loads without matching xgboost/sklearn versions)
_MODEL_JSON = M + 'day7_tuned_xgboost.json'
final.get_booster().save_model(_MODEL_JSON)
print('wrote', _MODEL_JSON)

try:
    from google.colab import files
    files.download(_MODEL_PKL)
    files.download(_MODEL_JSON)
except Exception as _e:
    print('download skipped:', _e)


wrote /content/drive/MyDrive/HMDA_Data/modelling/day7_tuned_xgboost.pkl
wrote /content/drive/MyDrive/HMDA_Data/modelling/day7_tuned_xgboost.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 4 - Compare to Day 6 baseline & save artifacts
Load Day 6's untuned XGBoost numbers for context, save `day7_tuned_metrics.csv` (tuned val + test), and plot the tuned model's `X_val` ROC/PR curves.

In [8]:
base = pd.read_csv(M + 'day6_model_metrics.csv', index_col=0)
day6_xgb = base.loc['XGBoost'] if 'XGBoost' in base.index else None
print('Day6 untuned XGBoost (X_val):')
print(day6_xgb.round(4) if day6_xgb is not None else 'n/a')

tuned = pd.DataFrame({
    'XGB_tuned_Xval':  {k: res_val[k] for k in ['roc_auc','pr_auc','precision','recall','f1']},
    'XGB_tuned_Xtest': {k: res_test[k] for k in ['roc_auc','pr_auc','precision','recall','f1']},
}).T
print('\nTuned XGBoost:')
print(tuned.round(4))
tuned.to_csv(M + 'day7_tuned_metrics.csv')
print('saved day7_tuned_metrics.csv')

plt.figure(figsize=(6,6))
fpr, tpr, _ = roc_curve(y_val, res_val['proba']); plt.plot(fpr, tpr, label=f"tuned (AUC={res_val['roc_auc']:.3f})")
plt.plot([0,1],[0,1],'k--'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend()
plt.title('ROC - tuned XGBoost (X_val)'); plt.savefig(FIG+'roc_tuned.png', dpi=120); plt.close()
plt.figure(figsize=(6,6))
prec, rec, _ = precision_recall_curve(y_val, res_val['proba']); plt.plot(rec, prec, label=f"tuned (AP={res_val['pr_auc']:.3f})")
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.legend()
plt.title('PR - tuned XGBoost (X_val)'); plt.savefig(FIG+'pr_tuned.png', dpi=120); plt.close()
print('saved overlays to', FIG)


Day6 untuned XGBoost (X_val):
n/a

Tuned XGBoost:
                 roc_auc  pr_auc  precision  recall      f1
XGB_tuned_Xval    0.8958  0.9543     0.9147  0.8620  0.8876
XGB_tuned_Xtest   0.8961  0.9544     0.9148  0.8623  0.8878
saved day7_tuned_metrics.csv
saved overlays to /content/drive/MyDrive/HMDA_Data/modelling/figures/day7/


## Step 5 - Summary
Writes `markdown/day7_tuning_summary.md` with the tuning config, best params, and the tuned-vs-baseline comparison. Surfaces Day 8 (SHAP) and Day 9 (fairness) next steps.

In [9]:
summary_lines = [
 "# Day 7 Hyperparameter Tuning - Summary", "",
 f"- Tuning target: XGBoost; objective = PR-AUC on X_val; trials = {N_TRIALS}; trial subsample = {TRIAL_SUB_N}.",
 f"- scale_pos_weight fixed = {spw:.4f}. Feature set = 78 (from day7_model_features.csv).",
 f"- Best PR-AUC (X_val) = {study.best_value:.4f}. Best params: {json.dumps(best_params)}",
 "- Day 6 untuned XGBoost (X_val): " + (day6_xgb.round(4).to_dict().__str__() if day6_xgb is not None else 'n/a'),
 f"- Tuned XGBoost X_val: ROC-AUC={res_val['roc_auc']:.4f} PR-AUC={res_val['pr_auc']:.4f} P={res_val['precision']:.4f} R={res_val['recall']:.4f} F1={res_val['f1']:.4f}",
 f"- Tuned XGBoost X_test (single look): ROC-AUC={res_test['roc_auc']:.4f} PR-AUC={res_test['pr_auc']:.4f} P={res_test['precision']:.4f} R={res_test['recall']:.4f} F1={res_test['f1']:.4f}",
 "- Note: X_test was used only for this final evaluation (tuning used X_val only).",
 "- Artifacts: day7_best_xgb_params.json, day7_tuned_metrics.csv, day7_optuna_progress.csv, figures/day7/*.png.",
 "- Next: Day 8 SHAP on tuned model; Day 9 fairness across derived_race/ethnicity/sex.",
]
summary = "\n".join(summary_lines)
print(summary)
summary_md = M + 'day7_tuning_summary.md'
open(summary_md, 'w').write(summary)
print('wrote', summary_md)
try:
    from google.colab import files
    for _f in [summary_md, M + 'day7_tuned_metrics.csv',
               M + 'day7_best_xgb_params.json', M + 'day7_optuna_progress.csv']:
        files.download(_f)
except Exception as _e:
    print('download skipped:', _e)


# Day 7 Hyperparameter Tuning - Summary

- Tuning target: XGBoost; objective = PR-AUC on X_val; trials = 10; trial subsample = 1000000.
- scale_pos_weight fixed = 0.3396. Feature set = 78 (from day7_model_features.csv).
- Best PR-AUC (X_val) = 0.9494. Best params: {"max_depth": 10, "learning_rate": 0.029393392247576217, "n_estimators": 3000, "subsample": 0.86637030299845, "colsample_bytree": 0.7602638701835958, "min_child_weight": 6, "reg_lambda": 0.012341225700071089, "reg_alpha": 0.06889775400590788}
- Day 6 untuned XGBoost (X_val): n/a
- Tuned XGBoost X_val: ROC-AUC=0.8958 PR-AUC=0.9543 P=0.9147 R=0.8620 F1=0.8876
- Tuned XGBoost X_test (single look): ROC-AUC=0.8961 PR-AUC=0.9544 P=0.9148 R=0.8623 F1=0.8878
- Note: X_test was used only for this final evaluation (tuning used X_val only).
- Artifacts: day7_best_xgb_params.json, day7_tuned_metrics.csv, day7_optuna_progress.csv, figures/day7/*.png.
- Next: Day 8 SHAP on tuned model; Day 9 fairness across derived_race/ethnicity/sex.
wrot

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Day 7 - Results summary (filled in after running)
After execution: best PR-AUC, best hyperparameters, and tuned-vs-Day6 comparison land in `day7_tuned_metrics.csv` and `markdown/day7_tuning_summary.md`.